# Reading the 40 by the yard

**NFL Big Data Bowl 2027 — companion notebook.** Everything in the writeup is produced by this
notebook from the competition files. It builds speed-by-distance curves from the 40-yard-dash
tracking, links each yard of the curve to regular-season outcomes, and runs the pass-rusher models.

Sections: 1 setup · 2 the 40 as a curve · 3 game outcomes · 4 which yards matter · 5 pass rushers ·
6 figures and tables.

In [ ]:
import os, warnings, numpy as np, pandas as pd, polars as pl
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

import glob
def find_data():
    env = os.environ.get("BDB_DATA")
    if env:
        return env
    for root in ("/kaggle/input", "../data", "data"):
        hits = glob.glob(f"{root}/**/players.csv", recursive=True)
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError("competition CSVs not found; set BDB_DATA")
DATA = find_data()
ON_KAGGLE = DATA.startswith("/kaggle")
OUT = os.environ.get("BDB_OUT", "/kaggle/working" if ON_KAGGLE else "../assets")
print("data:", DATA, "| output:", OUT)
os.makedirs(OUT, exist_ok=True)
NULLS = ["NA", ""]
rng = np.random.default_rng(2027)

# colours: one categorical palette, fixed order; text in ink, never in series colour
PAL = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "yellow": "#eda100",
     "magenta": "#e87ba4", "green": "#008300", "violet": "#4a3aa7", "red": "#e34948"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "figure.dpi": 110,
                     "savefig.dpi": 200, "savefig.bbox": "tight", "figure.facecolor": "white"})

def read(name, columns=None):
    return pl.read_csv(f"{DATA}/{name}.csv", null_values=NULLS, infer_schema_length=100_000, columns=columns)

players = read("players").to_pandas().set_index("nfl_id")
combine = read("combine_results").to_pandas().set_index("nfl_id")
games = read("games").to_pandas()
GRP = {"WR": "WR", "TE": "TE", "CB": "CB", "DB": "CB", "SS": "S", "FS": "S", "T": "OL", "G": "OL", "C": "OL",
       "DE": "DL", "DT": "DL", "NT": "DL", "OLB": "OLB", "ILB": "LB", "MLB": "LB", "RB": "RB", "FB": "RB"}
players["grp"] = players.nfl_position.map(GRP)
print(len(players), "prospects;", combine.forty.notna().sum(), "ran the 40")

## 2. The 40 as a curve
The tracking clock does not start on the timer, so we index speed by **distance covered** instead
of time. Speed at a given yard is independent of when the recording started. When a time is
needed, the official 40 pins the clock: time from 10 to 40 yards is the integral of 1/speed, and
the sensor 10-yard split is `forty − t(10→40)`.

In [ ]:
GRID_D = np.arange(0.0, 40.5, 0.5)

def speed_by_distance(g):
    g = g.sort_values("time")
    x, y = g.x.to_numpy(), g.y.to_numpy()
    s = pd.Series(g.s.to_numpy()).rolling(3, center=True, min_periods=1).mean().to_numpy()
    d = np.maximum.accumulate(np.hypot(x - x[0], y - y[0]))
    if d[-1] < 39:
        return None
    ok = np.concatenate([[True], np.diff(d) > 0])
    return np.interp(GRID_D, d[ok], s[ok], right=np.nan)

ct = read("combine_tracking").filter((pl.col("drill_type") == "FORTY_YARD_DASH") & (pl.col("entity_type") == "PLAYER")).to_pandas()
curves, runs = [], []
for ev, g in ct.groupby("event_id", sort=False):
    sd = speed_by_distance(g)
    if sd is None:
        continue
    nid = int(g.nfl_id.iloc[0])
    curves.append(pd.DataFrame({"event_id": ev, "nfl_id": nid, "attempt": g.attempt.iloc[0], "d": GRID_D, "s": sd}))
    with np.errstate(divide="ignore"):
        inv = 1 / np.where(sd > 0.3, sd, np.nan)
    cum = np.concatenate([[0], np.nancumsum(0.5 * (inv[1:] + inv[:-1]) * np.diff(GRID_D))])
    t10_40 = cum[-1] - cum[np.searchsorted(GRID_D, 10)]
    runs.append({"event_id": ev, "nfl_id": nid, "attempt": g.attempt.iloc[0],
                 "s_at_2.5": sd[5], "s_at_5": sd[10], "s_at_10": sd[20], "s_at_20": sd[40],
                 "top_speed": np.nanmax(sd), "t_10_40_sensor": t10_40,
                 "ten_sensor": combine.forty.get(nid, np.nan) - t10_40})
curves = pd.concat(curves, ignore_index=True)
runs = pd.DataFrame(runs)
PM = curves.groupby(["nfl_id", "d"]).s.mean().unstack().ffill(axis=1)   # player-mean curve (last half-yard carried if a run ends at 39.5)
P = runs.groupby("nfl_id")[["s_at_2.5", "s_at_5", "s_at_10", "s_at_20", "top_speed", "t_10_40_sensor", "ten_sensor"]].mean()
P["n_runs"] = runs.groupby("nfl_id").size()
P = P.join(combine[["combine_weight", "ten_yd_split", "forty", "three_cone", "short_shuttle"]]).join(players[["display_name", "nfl_position", "grp", "draft_year", "draft_overall_pick"]])
P["pick"] = P.draft_overall_pick.fillna(262)
VALD = runs.join(combine[["ten_yd_split", "forty"]], on="nfl_id")
val = {"r_ten": VALD.ten_sensor.corr(VALD.ten_yd_split), "r_10_40": VALD.t_10_40_sensor.corr(VALD.forty - VALD.ten_yd_split),
       "ten_bias": (VALD.ten_sensor - VALD.ten_yd_split).mean(), "ten_sd": (VALD.ten_sensor - VALD.ten_yd_split).std()}
print(f"{len(runs)} runs, {len(P)} players. Sensor 10-yd split vs official: r={val['r_ten']:.2f}, "
      f"bias={val['ten_bias']:+.3f}s, sd={val['ten_sd']:.3f}s. Sensor 10→40 vs official: r={val['r_10_40']:.2f}")

# test-retest reliability of speed at each yard (players with two runs)
W = curves.pivot_table(index=["nfl_id", "d"], columns="attempt", values="s").dropna(subset=[1, 2])
rel = W.groupby("d").apply(lambda g: g[1].corr(g[2]))
print("test-retest r by yard:", {d: round(rel[d], 2) for d in (1, 2.5, 5, 10, 20, 40)}, "n =", W.groupby("d").size().iloc[0])

# early burst beyond what the official 40 implies (within position group)
P["acc_resid"] = np.nan
for g, sub in P.dropna(subset=["forty", "s_at_5"]).groupby("grp"):
    P.loc[sub.index, "acc_resid"] = smf.ols("s_at_5 ~ forty", data=sub).fit().resid

## 3. Regular-season outcomes
* **Game speed**: 95th percentile of per-play max speed from game tracking (2023–25 REG).
* **Pass rushers** (DE/DT/NT/OLB at EDGE/INTERIOR): pressure rate and get-off, adjusted for
  alignment, down, distance and season.
* **Receivers**: separation at pass-forward on break routes, adjusted for route, coverage,
  cushion, pass length, alignment, down and season.
* **Offensive line**: pressure allowed, adjusted for spot, down, distance, rushers faced, season.

In [ ]:
reg = games.loc[games.season_type == "REG", ["game_id", "season"]]
play_max = pl.concat([pl.scan_csv(f"{DATA}/game_tracking_{y}.csv", null_values=NULLS, infer_schema_length=100_000)
                        .select("game_id", "play_id", "nfl_id", "s") for y in (2023, 2024, 2025)]) \
    .join(pl.from_pandas(reg).lazy(), on="game_id").group_by("nfl_id", "game_id", "play_id").agg(smax=pl.col("s").max()).collect().to_pandas()
speed = play_max.groupby("nfl_id").agg(plays=("smax", "size"), ig_p95=("smax", lambda s: s.quantile(.95)), ig_max=("smax", "max"))
P = P.join(speed)

pp = read("player_play").to_pandas().merge(reg, on="game_id").join(players[["nfl_position", "draft_year"]], on="nfl_id")
pp["long"] = (pp.yards_to_go >= 7).astype(int)

d = pp[pp.nfl_position.isin(["DE", "DT", "NT", "OLB"]) & pp.player_get_off.notna() & pp.lined_up_position.isin(["EDGE", "INTERIOR_LINE"])].copy()
d["pressure"] = d.time_to_pressure.notna().astype(int)
d["get_off_adj"] = smf.ols("player_get_off ~ C(lined_up_position) + C(down) + long + C(season)", data=d).fit().resid
dl = d.groupby("nfl_id").agg(rush_snaps=("play_id", "size"), get_off_adj=("get_off_adj", "mean"), pressure_rate=("pressure", "mean"),
                            sack_rate=("sack", lambda s: s.fillna(0).mean()), edge_share=("lined_up_position", lambda s: (s == "EDGE").mean()))
P = P.join(dl)

r = pp[pp.nfl_position.isin(["WR", "TE"]) & pp.route_ran.notna() & pp.separation_at_pass_forward.notna()
       & pp.team_coverage_man_zone.notna() & pp.lined_up_position.isin(["WR", "SLOT_WR", "TE"])].copy()
r["cushion_f"] = r.cushion.fillna(r.cushion.median())
r["sep_adj"] = smf.ols("separation_at_pass_forward ~ C(route_ran)*C(team_coverage_man_zone) + cushion_f + pass_length + C(season) + C(lined_up_position) + C(down)", data=r).fit().resid
r["brk"] = r.route_ran.isin(["HITCH", "OUT", "IN", "SLANT", "COMEBACK", "CORNER", "POST", "CROSS", "ANGLE"])
rec = r.groupby("nfl_id").agg(routes=("sep_adj", "size"), sep_break=("sep_adj", lambda s: s[r.loc[s.index, "brk"]].mean()),
                              sep_vert=("sep_adj", lambda s: s[~r.loc[s.index, "brk"]].mean()))
P = P.join(rec)

o = pp[pp.nfl_position.isin(["T", "G", "C"]) & pp.pressure_allowed.notna() & pp.lined_up_position.isin(["T", "G", "C"])].copy()
o["pa"] = o.pressure_allowed.astype(int)
o["pa_adj"] = o.pa - smf.logit("pa ~ C(lined_up_position) + C(down) + long + C(season) + pass_rushers_encountered", data=o).fit(disp=0).predict(o)
P = P.join(o.groupby("nfl_id").agg(pass_pro_snaps=("pa", "size"), pa_adj=("pa_adj", "mean")))

# reliability of the outcomes (split halves of snaps)
def split_half(df, key, col, minn):
    h = df.assign(h=rng.integers(0, 2, len(df))).groupby([key, "h"])[col].mean().unstack().dropna()
    n = df.groupby(key).size(); h = h[n.reindex(h.index) >= minn]
    return h[0].corr(h[1]), len(h)
print("split-half reliability: pressure", split_half(d, "nfl_id", "pressure", 50), "| get-off", split_half(d, "nfl_id", "get_off_adj", 50),
      "| receiver sep", split_half(r, "nfl_id", "sep_adj", 100), "| OL pressure allowed", split_half(o, "nfl_id", "pa_adj", 100))
P.to_csv(f"{OUT}/player_table.csv")

## 4. Which yards of the 40 carry the signal?
For every outcome we correlate speed at each half-yard of the curve with the outcome and
bootstrap a 95% band. The shape of that curve says *where* in the 40 the information lives.

In [ ]:
def corr_cols(X, y):
    Xc = X - X.mean(0); yc = y - y.mean()
    return (Xc * yc[:, None]).sum(0) / np.sqrt((Xc ** 2).sum(0) * (yc ** 2).sum())

def r_by_yard(ids, outcome, nboot=1000):
    sub = P.loc[ids].dropna(subset=[outcome]); X = PM.loc[sub.index].to_numpy(); y = sub[outcome].to_numpy()
    bs = np.array([corr_cols(X[i], y[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(nboot))])
    return pd.DataFrame({"d": PM.columns, "r": corr_cols(X, y), "lo": np.nanpercentile(bs, 2.5, 0), "hi": np.nanpercentile(bs, 97.5, 0), "n": len(y)})

rushers = P.index[P.grp.isin(["DL", "OLB"]) & (P.rush_snaps >= 50)]
receivers = P.index[P.grp.isin(["WR", "TE"]) & (P.routes >= 100)]
linemen = P.index[(P.grp == "OL") & (P.pass_pro_snaps >= 100)]
RBY = {"Pass rushers: pressure rate": r_by_yard(rushers, "pressure_rate"),
       "Pass rushers: sack rate": r_by_yard(rushers, "sack_rate"),
       "Receivers: separation, break routes": r_by_yard(receivers, "sep_break"),
       "Receivers: separation, vertical routes": r_by_yard(receivers, "sep_vert"),
       "Offensive line: pressure allowed": r_by_yard(linemen, "pa_adj")}
for g in ["WR", "DL", "CB", "S", "OL"]:
    RBY[f"{g}: game top speed"] = r_by_yard(P.index[(P.grp == g) & (P.plays >= 100)], "ig_p95")
for k, v in RBY.items():
    pk = v.iloc[v.r.abs().argmax()]
    print(f"{k:40s} n={int(v.n[0]):3d}  r@5yd={v.r[v.d == 5].iloc[0]:+.2f}  r@40yd={v.r[v.d == 40].iloc[0]:+.2f}  peak {pk.r:+.2f} at {pk.d:g} yd")

S = P[P.plays >= 100].copy(); S["used"] = S.ig_p95 / S.top_speed
usage = S.groupby("grp").agg(n=("used", "size"), combine_top=("top_speed", "mean"), game_p95=("ig_p95", "mean"), used=("used", "mean"),
                             r_within=("top_speed", lambda s: s.corr(S.loc[s.index, "ig_p95"]))).loc[["WR", "CB", "TE", "S", "DL", "OL"]]
print(usage.round(2))

## 5. Pass rushers: the first five yards
Speed at the 5-yard mark against pressure rate, controlling for the stopwatch (official 10 and
40), weight, draft slot and alignment; leave-one-draft-class-out cross-validation; effect size.

In [ ]:
D = P.loc[rushers].dropna(subset=["forty", "ten_yd_split"]).copy()
D["edge"] = (D.edge_share >= .5).astype(int); D["log_pick"] = np.log(D.pick)

def partial_r(x, ctrl, nboot=4000):
    cols = [c.strip() for c in ctrl.split("+")]
    Z = np.column_stack([np.ones(len(D)), D[cols].to_numpy(float)]); y = D.pressure_rate.to_numpy(); xv = D[x].to_numpy(float)
    def f(i):
        Zi = Z[i]; ry = y[i] - Zi @ np.linalg.lstsq(Zi, y[i], rcond=None)[0]; rx = xv[i] - Zi @ np.linalg.lstsq(Zi, xv[i], rcond=None)[0]
        return np.corrcoef(ry, rx)[0, 1]
    bs = [f(rng.integers(0, len(D), len(D))) for _ in range(nboot)]
    return f(np.arange(len(D))), np.percentile(bs, [2.5, 97.5])
CTRL = "ten_yd_split + forty + combine_weight + log_pick + edge"
pr_s5 = partial_r("s_at_5", CTRL); pr_res = partial_r("acc_resid", CTRL)
pr_forty = partial_r("forty", "combine_weight + log_pick + edge"); pr_ten = partial_r("ten_yd_split", "combine_weight + log_pick + edge")
print(f"n={len(D)}  partial r | stopwatch+weight+pick+edge: s_at_5 {pr_s5[0]:+.2f} {np.round(pr_s5[1],2)};  early burst beyond the 40 {pr_res[0]:+.2f} {np.round(pr_res[1],2)}")
print(f"for comparison, partial r | weight+pick+edge: forty {pr_forty[0]:+.2f} {np.round(pr_forty[1],2)};  ten {pr_ten[0]:+.2f} {np.round(pr_ten[1],2)}")

def cv_r2(cols):
    pred = pd.Series(index=D.index, dtype=float)
    for yr in D.draft_year.unique():
        tr, te = D[D.draft_year != yr], D[D.draft_year == yr]
        pred[te.index] = LinearRegression().fit(tr[cols], tr.pressure_rate, sample_weight=np.sqrt(tr.rush_snaps)).predict(te[cols])
    return 1 - ((D.pressure_rate - pred) ** 2).sum() / ((D.pressure_rate - D.pressure_rate.mean()) ** 2).sum()
BASE = ["log_pick", "combine_weight", "edge"]
CV = {"Draft slot, weight, alignment": cv_r2(BASE), "+ official 10-yard split": cv_r2(BASE + ["ten_yd_split"]),
      "+ official 40": cv_r2(BASE + ["forty"]), "+ both stopwatch times": cv_r2(BASE + ["ten_yd_split", "forty"]),
      "+ sensor speed at 5 yd": cv_r2(BASE + ["s_at_5"])}
print("leave-one-draft-class-out R²:", {k: round(v, 3) for k, v in CV.items()})
m = smf.ols("pressure_rate ~ s_at_5 + combine_weight + edge", data=D).fit(); sd5 = D.s_at_5.std()
eff = {"sd5": sd5, "dpress": m.params.s_at_5 * sd5, "p": m.pvalues.s_at_5, "mean": D.pressure_rate.mean(), "sd": D.pressure_rate.std()}
print(f"one SD of speed at 5 yd = {sd5:.2f} yd/s → pressure rate {eff['dpress']*100:+.1f} points (mean {eff['mean']*100:.1f}%, SD {eff['sd']*100:.1f}; p={eff['p']:.3f})")
med = smf.ols("get_off_adj ~ s_at_5 + combine_weight + edge", data=D).fit()
print(f"speed at 5 yd → in-game get-off: {med.params.s_at_5*sd5:+.3f} s per SD (p={med.pvalues.s_at_5:.2f}); get-off SD across players {D.get_off_adj.std():.3f} s")

## 6. Figures and tables

In [ ]:
POS_COL = {"WR": PAL["blue"], "DL": PAL["orange"], "OL": PAL["aqua"], "CB": PAL["yellow"], "S": PAL["magenta"], "TE": PAL["green"]}

# Figure 1: the curve, its validation and its reliability
fig, ax = plt.subplots(1, 3, figsize=(13, 3.9), gridspec_kw={"width_ratios": [1.5, 1, 1]})
ends = {}
for g in ["WR", "CB", "DL", "OL"]:
    ids = P.index[P.grp == g]; m_ = PM.loc[ids].mean(); q1, q3 = PM.loc[ids].quantile(.25), PM.loc[ids].quantile(.75)
    ax[0].fill_between(GRID_D, q1, q3, color=POS_COL[g], alpha=.12, lw=0); ax[0].plot(GRID_D, m_, color=POS_COL[g], lw=2); ends[g] = (m_.iloc[-1], len(ids))
ys = pd.Series({g: e[0] for g, e in ends.items()}).sort_values()
for k in range(1, len(ys)):                      # push labels apart by at least 0.45 yd/s
    ys.iloc[k] = max(ys.iloc[k], ys.iloc[k - 1] + 0.45)
for g, yy_ in ys.items():
    ax[0].text(40.8, yy_, f"{g} (n={ends[g][1]})", color=INK, va="center", fontsize=9)
ax[0].axvspan(0, 10, color="#f0efec", zorder=0); ax[0].text(5, 12.55, "acceleration\nphase", ha="center", va="top", color=INK2, fontsize=8.5)
ax[0].text(27, 12.55, "top-speed phase", ha="center", va="top", color=INK2, fontsize=8.5)
ax[0].set(xlim=(0, 47), ylim=(3, 12.8), xlabel="distance covered (yd)", ylabel="speed (yd/s)", title="A. The 40 as a speed-by-distance curve")
ax[1].scatter(VALD.ten_yd_split, VALD.ten_sensor, s=12, color=PAL["blue"], alpha=.55, lw=0)
ax[1].plot([1.4, 2.0], [1.4, 2.0], color=INK2, lw=1, ls="--"); ax[1].set(xlabel="official 10-yard split (s)", ylabel="sensor 10-yard split (s)", title=f"B. Sensor vs stopwatch (r = {val['r_ten']:.2f})")
ax[2].plot(rel.index, rel.values, color=PAL["blue"], lw=2); ax[2].set(xlabel="distance covered (yd)", ylabel="run-to-run correlation", ylim=(0, 1.02), title="C. Repeatability of speed at each yard")
ax[2].axvspan(0, 10, color="#f0efec", zorder=0)
for d_ in (2.5, 5, 10, 20): ax[2].annotate(f"{rel[d_]:.2f}", (d_, rel[d_]), textcoords="offset points", xytext=(4, -12), fontsize=8, color=INK2)
fig.savefig(f"{OUT}/fig1_curve.png"); plt.close(fig)

# Figure 2: correlation by yard, by outcome
fig, ax = plt.subplots(1, 3, figsize=(13, 3.9), sharey=True)
panels = [("A. Pass rushers (n=%d)", [("Pass rushers: pressure rate", PAL["orange"], "pressure rate"), ("Pass rushers: sack rate", PAL["yellow"], "sack rate")]),
          ("B. Receivers (n=%d)", [("Receivers: separation, break routes", PAL["blue"], "break routes"), ("Receivers: separation, vertical routes", PAL["violet"], "vertical routes")]),
          ("C. Offensive line (n=%d)", [("Offensive line: pressure allowed", PAL["aqua"], "pressure allowed")])]
for a, (title, series) in zip(ax, panels):
    for key, col, lab in series:
        v_ = RBY[key]; a.fill_between(v_.d, v_.lo, v_.hi, color=col, alpha=.12, lw=0); a.plot(v_.d, v_.r, color=col, lw=2); a.text(v_.d.iloc[-1] + .6, v_.r.iloc[-1], lab, color=INK, fontsize=8.5, va="center")
    a.axhline(0, color=INK2, lw=.8); a.axvspan(0, 10, color="#f0efec", zorder=0); a.set(title=title % int(RBY[series[0][0]].n[0]), xlabel="yard of the 40", xlim=(0, 52))
ax[0].set_ylabel("correlation with NFL outcome"); ax[0].set_ylim(-.6, .7); ax[1].text(.02, .04, "separation at pass-forward, adjusted", transform=ax[1].transAxes, fontsize=8, color=INK2)
fig.savefig(f"{OUT}/fig2_corr_by_yard.png"); plt.close(fig)

# Figure 3: how much of the Combine speed shows up in games
fig, ax = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.3, 1]})
for g in ["WR", "CB", "TE", "S", "DL", "OL"]:
    v_ = RBY.get(f"{g}: game top speed")
    if v_ is not None:
        ax[0].plot(v_.d, v_.r, color=POS_COL[g], lw=2); ax[0].text(40.6, v_.r.iloc[-1], g, color=INK, fontsize=9, va="center")
ax[0].axhline(0, color=INK2, lw=.8); ax[0].axvspan(0, 10, color="#f0efec", zorder=0)
ax[0].set(xlabel="yard of the 40", ylabel="correlation with game top speed (p95)", xlim=(0, 44), ylim=(-.5, .8), title="A. Which yard predicts game speed, by position")
yy = np.arange(len(usage))
ax[1].barh(yy, usage.used * 100, color=PAL["blue"], height=.55)
for i, (g, row) in enumerate(usage.iterrows()):
    ax[1].text(row.used * 100 + 1, i, f"{row.used*100:.0f}%  ·  r = {row.r_within:+.2f}", va="center", fontsize=9, color=INK)
ax[1].set(yticks=yy, yticklabels=[f"{g} (n={int(n)})" for g, n in usage.n.items()], xlim=(0, 115), xlabel="game top speed as % of Combine top speed", title="B. Share of Combine speed used on Sundays")
ax[1].invert_yaxis(); ax[1].grid(axis="y", visible=False)
fig.savefig(f"{OUT}/fig3_speed_used.png"); plt.close(fig)

# Figure 4: pass rushers
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"width_ratios": [1.3, 1], "wspace": 0.6})
sz = 18 + 60 * (D.rush_snaps / D.rush_snaps.max())
ax[0].scatter(D.s_at_5, D.pressure_rate * 100, s=sz, color=PAL["orange"], alpha=.6, lw=0)
xx = np.linspace(D.s_at_5.min(), D.s_at_5.max(), 50); mm = smf.ols("pressure_rate ~ s_at_5", data=D).fit()
ax[0].plot(xx, mm.predict(pd.DataFrame({"s_at_5": xx})) * 100, color=INK2, lw=1.2, ls="--")
lab = D.sort_values("pressure_rate").index[-2:].tolist() + D.sort_values("acc_resid").index[-2:].tolist() + D.sort_values("s_at_5").index[:1].tolist()
for k, nid in enumerate(lab):
    ax[0].annotate(D.display_name[nid], (D.s_at_5[nid], D.pressure_rate[nid] * 100), textcoords="offset points", xytext=(6, -9 if k % 2 else 4), fontsize=8, color=INK)
ax[0].set(xlabel="speed at the 5-yard mark of the 40 (yd/s)", ylabel="pressure rate (% of pass-rush snaps)", title=f"A. First five yards vs pressure rate (r = {D.s_at_5.corr(D.pressure_rate):.2f}, n = {len(D)})")
ax[0].text(.02, .97, "marker size = pass-rush snaps", transform=ax[0].transAxes, fontsize=8, color=INK2, va="top")
names = list(CV); vals = [CV[k] for k in names]; names = [n.replace("Draft slot, weight, alignment", "draft slot, weight,\nalignment (base)") for n in names]
ax[1].barh(range(len(names)), np.array(vals) * 100, color=[PAL["blue"] if "sensor" in n else "#b9b8b3" for n in names], height=.55)
for i, v_ in enumerate(vals): ax[1].text(v_ * 100 + .8, i, f"{v_*100:.0f}%", va="center", fontsize=9, color=INK)
ax[1].set(yticks=range(len(names)), yticklabels=names, xlabel="out-of-class R² for pressure rate (%)", title="B. Leave-one-draft-class-out validation", xlim=(0, 48))
ax[1].invert_yaxis(); ax[1].grid(axis="y", visible=False)
fig.savefig(f"{OUT}/fig4_rushers.png"); plt.close(fig)

# Table 1: rushers whose first five yards beat (or trailed) what their 40 time implied
tbl = D.assign(pressure=lambda x: (x.pressure_rate * 100).round(1), get_off=lambda x: x.get_off_adj.round(2), s5=lambda x: x.s_at_5.round(2),
               burst=lambda x: x.acc_resid.round(2), pick=lambda x: x.pick.astype(int))
cols = ["display_name", "nfl_position", "draft_year", "pick", "forty", "ten_yd_split", "s5", "burst", "pressure", "rush_snaps"]
top = tbl.sort_values("acc_resid", ascending=False).head(6)[cols]; bot = tbl.sort_values("acc_resid").head(4)[cols]
table1 = pd.concat([top, bot]); table1.to_csv(f"{OUT}/table1_rushers.csv", index=False)
print(table1.to_string(index=False))
print("mean pressure rate, top-6 burst:", round(top.pressure.mean(), 1), "| bottom-4:", round(bot.pressure.mean(), 1), "| all:", round(tbl.pressure.mean(), 1))
print("Done. Figures in", OUT)
try:
    from IPython.display import Image, display
    for f in ("fig1_curve.png", "fig2_corr_by_yard.png", "fig3_speed_used.png", "fig4_rushers.png"):
        display(Image(filename=f"{OUT}/{f}", width=1000))
except Exception:
    pass